# Café QLoRA training — Kaggle T4

User-run notebook; GPU training has not been tested locally. Set GPU T4 and Internet
on. Attach HF_TOKEN (write access) and WANDB_API_KEY using Add-ons → Secrets.
The pinned GitHub commit must be pushed before running. Keep this notebook and its
output private; exported Hub repositories also start private. Optional private
GitHub access uses GITHUB_TOKEN through an environment-only authorization header.

Only train and validation are loaded here. No test result selects hyperparameters.
Base model, dataset, dependencies and llama.cpp are pinned. All GPU imports live
in the Kaggle Python 3.11 worker; the notebook kernel only orchestrates processes.


In [ ]:
GITHUB_USERNAME = ""  # Your GitHub username; no token here.
REPO_NAME = "cafe-tool-agent"  # Match the repository you push.
GIT_REF = "PASTE_PHASE4_COMMIT"  # Exact 40-character commit from the Phase 4 report.
PRIVATE_GITHUB = False  # True requires an attached GITHUB_TOKEN Kaggle secret.
HF_USERNAME = ""  # Blank: derive the account name from HF_TOKEN.

IS_TRAINING = True
LORA_R = 16
TRAIN_FRACTION = 1.0
RUN_NAME = "main-r16-full"
TRAIN_BATCH_SIZE = 4
GRAD_ACCUM = 4
RESUME_CHECKPOINT = ""  # Optional checkpoint directory restored from a prior saved run.

In [ ]:
import base64
import json
import os
import re
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

if not re.fullmatch(r"[a-f0-9]{40}", GIT_REF):
    raise ValueError("Set GIT_REF to the Phase 4 commit from Codex's report before running")
if not re.fullmatch(r"[A-Za-z0-9-]+", GITHUB_USERNAME):
    raise ValueError("Fill in GITHUB_USERNAME")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", REPO_NAME):
    raise ValueError("Invalid repository name")
secrets = UserSecretsClient()
ENV = os.environ.copy()
ENV["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
if IS_TRAINING:
    ENV["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
if PRIVATE_GITHUB:
    github_token = secrets.get_secret("GITHUB_TOKEN")
    credential = base64.b64encode(("x-access-token:" + github_token).encode()).decode()
    ENV.update(
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + credential,
    )
    del github_token, credential
ENV.update(
    CUDA_VISIBLE_DEVICES="0",
    TOKENIZERS_PARALLELISM="false",
    PYTHONHASHSEED="42",
    HF_HOME="/tmp/cafe-hf",
    VLLM_USE_V1="0",
    VLLM_ATTENTION_BACKEND="XFORMERS",
    CUBLAS_WORKSPACE_CONFIG=":4096:8",
)
# No credential appears in a command, URL, saved manifest or printed environment.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "uv==0.11.26"], check=True
)
UV = [sys.executable, "-m", "uv"]
REQUIREMENTS = [
    "torch==2.6.0",
    "transformers==4.51.3",
    "accelerate==1.6.0",
    "peft==0.15.2",
    "trl==0.17.0",
    "bitsandbytes==0.45.5",
    "datasets==3.5.0",
    "wandb==0.19.10",
    "safetensors==0.5.3",
    "numpy==1.26.4",
    "sentencepiece==0.2.0",
    "protobuf==5.29.4",
    "tiktoken==0.9.0",
    "tokenizers==0.21.4",
    "opencv-python-headless==4.11.0.86",
]
PIN_FILE = Path("/tmp/cafe-notebook-requirements.txt")
PIN_FILE.write_text("\n".join(REQUIREMENTS) + "\n")
PACKAGE = f"git+https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git@{GIT_REF}"


def make_environment(location):
    python = str(Path(location) / "bin/python")
    if not Path(python).is_file():
        subprocess.run(UV + ["venv", "--python", "3.11", str(location)], check=True, env=ENV)
    subprocess.run(
        UV
        + [
            "pip",
            "install",
            "--python",
            python,
            "--exclude-newer",
            "2026-09-30T00:00:00Z",
            "-r",
            str(PIN_FILE),
            PACKAGE,
        ],
        check=True,
        env=ENV,
    )
    return python


PYTHON = make_environment("/tmp/cafe-notebook-env")
if not HF_USERNAME:
    HF_USERNAME = subprocess.check_output(
        [
            PYTHON,
            "-c",
            "import os; from huggingface_hub import HfApi; "
            "print(HfApi(token=os.environ['HF_TOKEN']).whoami()['name'])",
        ],
        env=ENV,
        text=True,
    ).strip()
DATASET_REPO = HF_USERNAME + "/cafe-tool-calls"
print("Python 3.11 environment ready; credentials remain in process memory.")

## Train and measure T4 memory

Uses NF4 double quantization, float16, all seven attention/MLP projections,
assistant-only labels and gradient checkpointing. Default batch 4 × accumulation 4.
A forward/backward memory probe uses the full 4,096-token padded shape before any
optimizer step; its peak excludes optimizer state. The actual training peak is
reported separately. If the probe OOMs, choose batch 1 / accumulation 16, give the
run a new name and rerun; full conversations are retained. Do not lower the token
limit or silently truncate. Evaluation/save occur approximately every half epoch;
the best validation-loss checkpoint is exported. The W&B project is cafe-tool-agent.

Checkpoints are resumable: save the Kaggle output, attach it as a private input in
a new session and set RESUME_CHECKPOINT to its checkpoint directory. Keep the same
Git commit, dataset, parameters and run name. Kaggle session/GPU quota may require
more than one session; do not treat an interrupted training run as complete.


In [ ]:
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("RUN_NAME must be a simple directory name")
OUTPUT = Path("/kaggle/working") / ("cafe-" + RUN_NAME)
OUTPUT.mkdir(exist_ok=True)
subprocess.run(
    [
        PYTHON,
        "-m",
        "cafe_agent.kaggle.preflight",
        "--dataset-repo",
        DATASET_REPO,
        "--output",
        str(OUTPUT / "preflight.json"),
    ],
    check=True,
    env=ENV,
)
command = [
    PYTHON,
    "-m",
    "cafe_agent.kaggle.train",
    "--dataset-repo",
    DATASET_REPO,
    "--output",
    str(OUTPUT),
    "--run-name",
    RUN_NAME,
    "--lora-r",
    str(LORA_R),
    "--train-fraction",
    str(TRAIN_FRACTION),
    "--batch-size",
    str(TRAIN_BATCH_SIZE),
    "--accumulation",
    str(GRAD_ACCUM),
]
if RESUME_CHECKPOINT:
    command += ["--resume", RESUME_CHECKPOINT]
subprocess.run(command, check=True, env=ENV)

## Export and upload

Merge the best adapter into a fresh fp16 base on CPU. Build the pinned llama.cpp
with CUDA disabled; convert to f16 GGUF and quantize Q4_K_M. Upload adapter, merged
fp16 and quantized model, each with a model card and actual training metadata.
GPU-worker hours exclude notebook setup and CPU export and are not a billing receipt.


In [ ]:
# Training was a separate process and has released its GPU memory.
# CPU-only merge, llama.cpp compilation/conversion, then private Hub uploads.
subprocess.run(
    [
        PYTHON,
        "-m",
        "cafe_agent.kaggle.export",
        "--output",
        str(OUTPUT),
        "--namespace",
        HF_USERNAME,
        "--dataset-repo",
        DATASET_REPO,
    ],
    check=True,
    env={**ENV, "CUDA_VISIBLE_DEVICES": ""},
)

In [ ]:
summary = json.loads((OUTPUT / "export_summary.json").read_text())
print(json.dumps(summary, indent=2))
print("Copy this exact MERGED_REVISION into eval_gpu_kaggle.ipynb:")
print(summary["hub"]["merged"]["revision"])
print("Retain the saved version's Output and the W&B link.")
print("For an interactive run, download needed working files before ending the session.")